# 04 · 법원 등기정보광장 — 동산·채권담보등기 통계

| | |
|---|---|
| **목적** | 2026-08-12 실수집한 IROS 통계 4종의 EDA. 기능 12(S18)의 `residual_risk` 배경 근거 |
| **input** | `data/iros/*.json` 4개 (읽기 전용 · `data/fetch_iros.py` 산출) |
| **output** | `data-analysis/data/processed/iros_collateral_stats.parquet`<br>`data-analysis/reports/iros_collateral_stats_decisions.json` |

> 🚨 **인용 금지선이 데이터에 박혀 있다.** 각 JSON 의 `citation_guard` 필드를 산출물까지
> 그대로 실어 나른다 — 동산·채권 **합산** 통계라 "기계설비 담보 N건" 으로 인용하면 사실 왜곡이다.

In [ ]:
import sys
sys.path.insert(0, "..")
import json
import pandas as pd
from common import IROS, profile, save_decisions, save_processed

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)


## 1. 개요

| 항목 | 값 |
|---|---|
| 출처 | [data.iros.go.kr](https://data.iros.go.kr/rp/oa/openOapiIntro.do) — 대법원 등기정보광장 |
| 수집일 | **2026-08-12** (`retrieved_at` 필드 기준) |
| 수집 스크립트 | `data/fetch_iros.py` |
| 조회 구간 | 202308~202312 · 202401~202412 · 202501~202512 · 202601~202607 (4구간) |
| 한도 | 일 1,000회 · 1회 최대 1,000건 (초과 시 3개월 키 차단) |

**왜 파일 스냅샷인가** — D60(원본은 파일이 정본). 요청마다 외부 기관을 호출하면
"서명 시점 스냅샷 재현" 전제가 깨진다.

In [ ]:
files = sorted(IROS.glob("*.json"))
raw = {}
for p in files:
    raw[p.stem] = json.loads(p.read_text(encoding="utf-8"))
    print(f"{p.name:28} {p.stat().st_size:>7,} bytes")

### 1-1. 서비스 4종 메타

In [ ]:
meta = pd.DataFrame([
    {
        "slug": d["slug"],
        "service_id": d["service_id"],
        "서비스명": d["service_label"],
        "record_count": d["record_count"],
        "tot_unit": d["tot_unit"],
        "fields": ", ".join(d["fields"]) or "(없음)",
        "retrieved_at": d["retrieved_at"][:19],
    }
    for d in raw.values()
])
meta

## 2. 로드 + 기초 통계

`records` 를 하나의 long 포맷 테이블로 합친다. 다만 **합산은 하지 않는다** — 이유는 §3-1.

In [ ]:
rows = []
for d in raw.values():
    for r in d["records"]:
        rows.append({
            "slug": d["slug"],
            "서비스명": d["service_label"],
            "tot_unit": d["tot_unit"],
            **r,
        })
df_raw = pd.DataFrame(rows)
print(f"원본 shape: {df_raw.shape[0]:,}행 × {df_raw.shape[1]}열")
print("서비스별 행 수:")
print(df_raw["slug"].value_counts().to_string())
print(f"\n합계 {len(df_raw)}행 — 문서가 주장하는 291행과 일치하는지 확인")

In [ ]:
profile(df_raw)

In [ ]:
df_raw.head(5)

## 3. EDA

### 3-1. 🚨 `tot` 의 단위가 서비스마다 다르다

기관 명세에는 4종 모두 "결과 건수"로 적혀 있었지만, **실호출로 확인하니 누적채권액만 `원`** 이다.
모르고 합산하면 **21조를 건수에 더하게 된다.**

In [ ]:
print(meta[["slug", "서비스명", "tot_unit"]].to_string(index=False))
print()
df_raw["tot_num"] = pd.to_numeric(df_raw["tot"], errors="coerce")
print("단위별 tot 최댓값 —")
for unit, g in df_raw.groupby("tot_unit"):
    print(f"  {unit}: max={g['tot_num'].max():>20,.0f}   (slug: {sorted(g['slug'].unique())})")
print()
print("→ ⛔ tot 을 서비스 구분 없이 sum() 하면 안 된다.")

### 3-2. ⚠️ `apply_by_amount` 는 0행 — 기관 측 결함으로 확정

채권최고액별 신청현황(`0000000239`)만 `record_count = 0` 이다. **수집 실패가 아니다.**

`empty_windows` 가 **비어 있다는 점**이 핵심이다. `fetch_iros.py` 는 API 가 "조회 결과 없음"
코드를 주면 그 구간을 `empty_windows` 에 적는데(L149~152), 239 는 거기 안 걸렸다 —
**정상 응답으로 레코드를 주긴 줬다.** 다만 그 레코드가 **전 필드 빈 문자열**이라
수집기가 걸러 냈다(L138~139). 즉 *"응답은 200 인데 알맹이가 없다"* 는 뜻이고,
재시도로 해결되지 않는다.

In [ ]:
d = raw["apply_by_amount"]
print("service_id      :", d["service_id"])
print("record_count    :", d["record_count"])
print("fields          :", d["fields"])
print("windows 조회    :", len(d["windows"]), "구간", d["windows"])
print("empty_windows   :", d["empty_windows"], "← 비어 있다")
print()
print("두 경우를 구분해야 한다 —")
print("  empty_windows 에 구간이 있다  = API 가 '결과 없음' 코드를 반환 (진짜 빈 구간)")
print("  empty_windows 가 비었는데 0행 = 정상 응답인데 레코드가 전 필드 공백 → 수집기가 필터")
print()
print("239 는 후자다. 기관이 빈 껍데기를 정상 응답으로 내보내고 있다.")

### 3-3. 설정자 유형(`srprsCls`) 분포

In [ ]:
for slug in ["apply_cases", "status_by_grantor", "debt_by_grantor"]:
    g = df_raw[df_raw["slug"] == slug]
    unit = g["tot_unit"].iloc[0]
    print(f"── {slug} (단위: {unit})")
    agg = g.groupby("srprsCls")["tot_num"].sum().sort_values(ascending=False)
    for k, v in agg.items():
        print(f"   {k:22} {v:>22,.0f} {unit}")
    print()

### 3-4. 시계열 — 동산담보 신청이 연 몇 건 규모인가

In [ ]:
ac = df_raw[df_raw["slug"] == "apply_cases"].copy()
ac["연"] = ac["resDate"].str[:4]
연별 = ac.groupby("연")["tot_num"].sum()
print("동산·채권담보 신청사건 (단위: 건)")
for y, v in 연별.items():
    완전 = "" if y in ("2023", "2026") else "  ← 12개월 완전"
    print(f"  {y}: {v:>9,.0f}{완전}")
print()
print("⚠ 2023 은 8~12월(5개월) · 2026 은 1~7월(7개월)이라 연 합계로 비교하면 안 된다.")

### 3-5. 🚨 결측 패턴 — `srCls` 빈 문자열

`status_by_grantor` 만 `srCls` 필드를 갖는데 **전부 빈 문자열**이다.
NaN 이 아니라 `''` 라서 `isna()` 로는 안 잡힌다 — §2 프로파일의 `null%` 가 0 으로 나온 이유다.

In [ ]:
sc = df_raw["srCls"]
print("srCls null 수      :", sc.isna().sum(), "(다른 서비스엔 필드 자체가 없어 NaN)")
present = df_raw[df_raw["slug"] == "status_by_grantor"]["srCls"]
print("status_by_grantor 내 srCls 고유값:", [repr(v) for v in present.unique()])
print("→ 전부 빈 문자열. 정보량 0 이므로 사용하지 않는다.")

## 4. 전처리

`tot` 을 숫자로 바꾸고, **단위가 다르므로 컬럼을 분리**한다 — 같은 컬럼에 두면 언젠가 합산된다.

In [ ]:
df = df_raw.copy()
df["tot_건수"] = df["tot_num"].where(df["tot_unit"] == "건")
df["tot_원"] = df["tot_num"].where(df["tot_unit"] == "원")
df["연"] = df["resDate"].str[:4]
df["월"] = df["resDate"].str[5:7]

# srCls 는 전부 빈 문자열 → 제거
df = df.drop(columns=["srCls", "tot_num"])

# 인용 금지선을 데이터에 동봉한다 (모든 행에 같은 값)
df["citation_guard"] = raw["apply_cases"]["citation_guard"]

print(f"정제 후: {df.shape[0]:,}행 × {df.shape[1]}열")
print("컬럼:", list(df.columns))

## 5. 결정 로그

In [ ]:
column_decisions = [
    {"column": "slug", "used": True, "reason": "서비스 식별자. 단위·의미가 서비스마다 달라 반드시 함께 봐야 한다"},
    {"column": "서비스명", "used": True, "reason": "사람이 읽는 라벨"},
    {"column": "tot_unit", "used": True, "reason": "🚨 단위 구분. 이게 없으면 21조(원)를 건수에 더하게 된다"},
    {"column": "resDate", "used": True, "reason": "월 단위 시점. 연/월로 분해"},
    {"column": "srprsCls", "used": True, "reason": "담보권설정자 유형 — 법인/개인사업자 분포가 유일한 변별 축"},
    {"column": "tot", "used": True, "reason": "값. 문자열로 와서 숫자 변환 + 단위별 컬럼 분리"},
    {"column": "srCls", "used": False,
     "reason": "status_by_grantor 108행 전부 빈 문자열(''). NaN 이 아니라 isna() 로 안 잡힌다 — 정보량 0"},
]
print(f"컬럼 결정 {len(column_decisions)}건 / 원본 컬럼 {df_raw.shape[1] - 1}개(tot_num 파생 제외)")

In [ ]:
save_decisions(
    "iros_collateral_stats",
    column_decisions,
    source_url="https://data.iros.go.kr/rp/oa/openOapiIntro.do",
    collected_at="2026-08-12",
    fetch_script="data/fetch_iros.py",
    services={d["slug"]: {"service_id": d["service_id"], "label": d["service_label"],
                          "record_count": d["record_count"], "tot_unit": d["tot_unit"]}
              for d in raw.values()},
    row_count_raw=int(df_raw.shape[0]),
    row_count_processed=int(df.shape[0]),
    generated_columns=["tot_건수", "tot_원", "연", "월"],
    citation_guard=raw["apply_cases"]["citation_guard"],
    caveats=[
        "⛔ '기계설비 담보가 연 N건' 으로 인용 금지 — 동산·채권 합산이고 기계설비 비중은 비공개",
        "tot 의 단위가 서비스마다 다르다 (건 vs 원). 서비스 구분 없이 합산 불가",
        "apply_by_amount(0000000239) 는 0행 — 정상 응답에 전 필드 공백 레코드를 실어 보낸다. "
        "empty_windows 가 비어 있다는 게 그 증거이고, 재시도로 해결되지 않는다",
        "2023 은 8~12월 · 2026 은 1~7월 부분 구간이라 연 합계 비교 불가",
        "개별 물건의 담보 설정 여부는 조회 불가 — 집계 통계뿐 (등기부 열람은 유료 + 본인확인)",
    ],
)

## 6. 정제 데이터 저장

In [ ]:
save_processed("iros_collateral_stats", df)